<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 8 </h1></center>
<center><h2> Un RAG from scratch </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 8 · 25 à 30 min · ≈ 0,08 $ · prépare le Lab 4 et le projet**
> Avant la séance : `make corpus`, `make install-demo`, puis lancer une fois (reranker de 2,2 Go). Reranker : ≈ 1 s par question sur GPU, ≈ 15 s sur CPU.

![](assets/08_rag_pipeline.png)

In [1]:
import os
import re
import time
from pathlib import Path

# Move to the repository root when Jupyter starts the kernel in the demo folder.
if os.path.basename(os.getcwd()) == "demo":
    os.chdir("../..")

import numpy as np
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

load_dotenv()
gateway = {"base_url": os.environ["LLM_BASE_URL"], "api_key": os.environ["LLM_API_KEY"]}
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, **gateway)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", check_embedding_ctx_length=False, **gateway)

DOC_IDS = ["F2258", "F12410", "F2262", "F18260", "F33359", "F682", "F2405", "F34151", "F34154", "F35134",
           "F2278", "F2266", "F151", "F2381", "F2382", "F12803", "F2931", "F1016", "F2265", "F3156"]
docs = {}
for doc_id in DOC_IDS:
    docs[doc_id] = Path(f"data/corpus/{doc_id}.md").read_text(encoding="utf-8")
print(len(docs), "fiches chargées")

20 fiches chargées


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Un RAG naïf en direct </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> ```
> ┌─────────────────────────────────────────────────────────────┐
> │  Les quatre étapes d'un RAG (sans base vectorielle)         │
> ├─────────────────────────────────────────────────────────────┤
> │                                                             │
> │  1. DÉCOUPER    →  Fragmenter les documents                │
> │          ↓                                                  │
> │  2. VECTORISER  →  Transformer en embeddings               │
> │          ↓                                                  │
> │  3. CHERCHER    →  Trouver les fragments pertinents        │
> │          ↓                                                  │
> │  4. GÉNÉRER     →  Produire la réponse finale              │
> │                                                             │
> └─────────────────────────────────────────────────────────────┘
> ```

In [2]:
# splitter definition
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", # encoding used by GPT-4, GPT-3.5-turbo and text-embedding-ada-002 models
    chunk_size=300, # maximum chunk size in tokens
    chunk_overlap=0) # number of overlapping tokens between two chunks

# splitting text
chunks = []
for doc_id, text in docs.items():
    for piece in splitter.split_text(text):
        chunks.append({"doc_id": doc_id, "text": piece})

# embedding chunks
vectors = np.array(embeddings.embed_documents([chunk["text"] for chunk in chunks]))
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
print(len(chunks), "chunks, matrix", vectors.shape)

190 chunks, matrix (190, 1536)



Maintenant que nous avons créé nos vecteurs de embeddings, nous pouvons mettre en place notre système de RAG (les parties recherche et génération).

In [3]:
# Defining search function to output positions of k (here 3) closest chunks
def search(question, k=3):
    """Returns the positions of the k chunks closest to the question."""
    query = np.array(embeddings.embed_query(question))
    scores = vectors @ (query / np.linalg.norm(query))
    return list(np.argsort(-scores)[:k])

# Building prompt for the model
PROMPT = """Répondez à la question uniquement à partir des extraits ci-dessous, en citant l'identifiant de la fiche entre crochets.
Si les extraits ne contiennent pas la réponse, répondez : « Je ne sais pas. »

{context}

Question : {question}"""


def answer(question, positions):
    """Asks the model to answer from the given chunks."""
    context = ""
    for position in positions:
        context += f"[{chunks[position]['doc_id']}] {chunks[position]['text']}\n\n"
    return llm.invoke(PROMPT.format(context=context, question=question)).content


question = "Combien de jours de congé pour mon mariage ?"
print(answer(question, search(question)))

La durée de votre congé est de 4 jours. [F34154]


- **Une quarantaine de lignes**, une réponse juste **et sourcée**.
- Au projet, **Qdrant et LangChain** feront la même chose.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Une question où il échoue </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> Même RAG, autre question. On regarde ce que la recherche a remonté avant de lire la réponse.

In [4]:
QUESTION = "Combien de jours de congé pour enfant malade ?"
NEEDLE = "3 jours par an"


def rank_of_answer(positions, needle=NEEDLE):
    """Rank of the first chunk that contains the expected answer, or None."""
    for rank, position in enumerate(positions, start=1):
        if needle.lower() in chunks[position]["text"].lower():
            return rank
    return None


def show(answer_text):
    """Prints the answer, then a verdict readable from the back of the room."""
    print(answer_text)
    if "3 jours" in answer_text:
        print("\n✅ JUSTE : la règle générale (3 jours par an) est dans la réponse")
    else:
        print("\n❌ FAUX : la règle générale (3 jours par an) manque")


dense_top3 = search(QUESTION)
for position in dense_top3:
    print(chunks[position]["doc_id"], "|", chunks[position]["text"][:120].replace("\n", " "), "…")
print()
print("Rang du passage qui contient la réponse :", rank_of_answer(search(QUESTION, k=20)))
print()
show(answer(QUESTION, dense_top3))

F151 | Le salarié doit transmettre à son employeur au plus vite le certificat médical (ou une copie) constatant la maladie ou l …
F151 | Le congé pour enfant malade, le congé de présence parentale (voir F1631) et le congé pour l’annonce du handicap ou d’une …
F151 | Toutefois, des dispositions conventionnelles plus favorables peuvent prévoir une rémunération durant le congé.  La conve …



Rang du passage qui contient la réponse : 5



La durée maximale du congé pour enfant malade est fixée à 5 jours par an. [F151]

❌ FAUX : la règle générale (3 jours par an) manque


![](assets/08_chunk_split.png)

- Bonne fiche (F151), **mauvais passage** : le chunk de la réponse est **5e**, hors des 3 envoyés.
- Le modèle répond **« 5 jours »** : un cas particulier présenté comme la règle. **Faux, sourcé, sûr de lui.**
- **L'échec n° 1 d'un RAG vient de la recherche, pas de la génération.**
- **Avant d'accuser le modèle, une question : la preuve est-elle dans ce qu'il a reçu ?** Un test d'une ligne : `rank_of_answer`.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Réparer avec le titre </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Un chunk **perd son titre** : « 3 jours par an » ne dit pas **de quel congé** il parle.
- Réparation **gratuite** : on écrit le **chemin des titres devant le chunk**, **avant** la vectorisation.

```
avant :  « Autre cas … 3 jours par an »
après :  « [Congé enfant malade > Durée > Autre cas] … 3 jours par an »
```

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

by_title = MarkdownHeaderTextSplitter([("#", "title1"), ("##", "title2"), ("###", "title3")])

titled_chunks = []
for doc_id, text in docs.items():
    for section in by_title.split_text(text):
        path = " > ".join(section.metadata[key] for key in ("title1", "title2", "title3") if key in section.metadata)
        for piece in splitter.split_text(section.page_content):
            titled_chunks.append({"doc_id": doc_id, "path": path, "text": piece})

titled_vectors = np.array(embeddings.embed_documents([f"[{chunk['path']}]\n{chunk['text']}" for chunk in titled_chunks]))
titled_vectors = titled_vectors / np.linalg.norm(titled_vectors, axis=1, keepdims=True)
print(len(titled_chunks), "chunks avec titre, matrice", titled_vectors.shape)

277 chunks avec titre, matrice (277, 1536)


In [6]:
def search_titled(question, k=3):
    """Same search, on the vectors that include the title."""
    query = np.array(embeddings.embed_query(question))
    scores = titled_vectors @ (query / np.linalg.norm(query))
    return list(np.argsort(-scores)[:k])


def rank_of_titled(positions, needle=NEEDLE):
    """Rank of the first chunk whose BODY contains the expected answer (the title is not counted)."""
    for rank, position in enumerate(positions, start=1):
        if needle.lower() in titled_chunks[position]["text"].lower():
            return rank
    return None


def answer_titled(question, positions):
    """Asks the model to answer from titled chunks."""
    context = ""
    for position in positions:
        chunk = titled_chunks[position]
        context += f"[{chunk['doc_id']}] {chunk['path']}\n{chunk['text']}\n\n"
    return llm.invoke(PROMPT.format(context=context, question=question)).content


print("Rang de la réponse, sans titre :", rank_of_answer(search(QUESTION, k=20)))
print("Rang de la réponse, avec titre  :", rank_of_titled(search_titled(QUESTION, k=20)))

Rang de la réponse, sans titre : 5


Rang de la réponse, avec titre  : 3


- Rang de la réponse **5 → 3** : le chunk est enfin dans les 3 envoyés.
- Un cas ne prouve rien : on mesure sur **14 questions**.

In [7]:
TEST_SET = [
    ("Que prévoit l'article L3142-4 du Code du travail ?", "L3142-4"),
    ("Combien de jours de congés payés acquiert-on par mois travaillé ?", "2,5 jours"),
    ("Combien de jours de congé pour mon mariage ?", "4 jours"),
    ("Combien de jours de congé si mon père décède ?", "3 jours"),
    ("Mon employeur peut-il m'imposer mes dates de congés ?", "ordre des départs"),
    ("Quelle est la période légale de prise des congés payés ?", "1er mai au 31 octobre"),
    ("Le salarié peut-il travailler pendant ses congés payés ?", "travailler pendant"),
    ("Combien de jours pour la naissance de mon enfant ?", "3 jours"),
    ("Que se passe-t-il si un jour férié tombe pendant mes congés ?", "férié"),
    ("Combien de temps dure le congé sabbatique ?", "6 mois"),
    ("Quelle ancienneté faut-il pour un congé sabbatique ?", "36 mois"),
    ("Combien de jours de congé pour enfant malade ?", "3 jours par an"),
    ("Le congé pour enfant malade est-il payé ?", "pas rémunéré"),
    ("Combien de jours de congé pour le mariage de mon enfant ?", "1 jour"),
]


dense_hits = titled_hits = 0
for question, needle in TEST_SET:
    before = rank_of_answer(search(question, k=20), needle)
    after = rank_of_titled(search_titled(question, k=20), needle)
    if (before or 99) <= 3:
        dense_hits += 1
    if (after or 99) <= 3:
        titled_hits += 1
    print(f"sans titre {str(before):>4}   avec titre {str(after):>4}   {question}")
print("\nBon chunk dans les 3 premiers : sans titre", dense_hits, "/ 14 | avec titre", titled_hits, "/ 14")

sans titre    1   avec titre    1   Que prévoit l'article L3142-4 du Code du travail ?


sans titre    1   avec titre    1   Combien de jours de congés payés acquiert-on par mois travaillé ?


sans titre    1   avec titre    1   Combien de jours de congé pour mon mariage ?


sans titre    1   avec titre    1   Combien de jours de congé si mon père décède ?


sans titre    1   avec titre    1   Mon employeur peut-il m'imposer mes dates de congés ?


sans titre    3   avec titre    1   Quelle est la période légale de prise des congés payés ?


... (4 outputs left out)

sans titre    1   avec titre    1   Quelle ancienneté faut-il pour un congé sabbatique ?


sans titre    5   avec titre    3   Combien de jours de congé pour enfant malade ?


sans titre    5   avec titre    1   Le congé pour enfant malade est-il payé ?


sans titre    1   avec titre    1   Combien de jours de congé pour le mariage de mon enfant ?

Bon chunk dans les 3 premiers : sans titre 12 / 14 | avec titre 13 / 14


- Top 3 : **sans titre 12/14 → avec titre 13/14**.
- **3 questions améliorées**, **1 perdue**, 10 inchangées.
- Le titre est **gratuit** : aucun appel au modèle, juste du texte de plus dans la vectorisation.
- ⚠️ Mesure **stricte** : la preuve doit être dans le **corps** du chunk. La question perdue (`travailler pendant`) avait sa preuve dans un titre.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Réparer avec la section entière </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Une question **plus précise** : l'enfant a 8 ans, deux enfants à charge.
- Le titre ne suffit pas toujours : les trois cas (5, 5 et 3 jours) sont dans **trois chunks voisins**.
- Réparation : on cherche avec de **petits chunks**, on envoie la **section entière** de chacun.

```
chunk trouvé  ──▶  sa section (tous les chunks voisins)  ──▶  envoyé au modèle
```

In [8]:
QUESTION_2 = "Mon fils de 8 ans a la grippe et j'ai deux enfants à charge. Combien de jours par an puis-je m'absenter pour le garder ?"


def section_of(chunk):
    """The section of a chunk: the fiche and the first two levels of its title path."""
    return chunk["doc_id"], " > ".join(chunk["path"].split(" > ")[:2])


def expand_to_section(positions):
    """Positions of all the chunks that share a section with one of the given chunks."""
    found = [section_of(titled_chunks[position]) for position in positions]
    return [i for i, chunk in enumerate(titled_chunks) if section_of(chunk) in found]


def tokens_sent(positions):
    """Rough size of what is sent to the model (4 characters per token)."""
    return sum(len(titled_chunks[position]["text"]) for position in positions) // 4


print("Rang de la réponse, sans titre :", rank_of_answer(search(QUESTION_2, k=20)))
print("Rang de la réponse, avec titre  :", rank_of_titled(search_titled(QUESTION_2, k=20)))
print()

print("### Sans titre")
show(answer(QUESTION_2, search(QUESTION_2)))
print("\n### Avec le titre")
show(answer_titled(QUESTION_2, search_titled(QUESTION_2)))
print("\n### Avec le titre et la section entière")
section_positions = expand_to_section(search_titled(QUESTION_2))
show(answer_titled(QUESTION_2, section_positions))
print("\nChunks envoyés :", len(search_titled(QUESTION_2)), "→", len(section_positions), "| ≈", tokens_sent(section_positions), "tokens de texte")

Rang de la réponse, sans titre : None


Rang de la réponse, avec titre  : 5

### Sans titre


La durée maximale du congé pour enfant malade est fixée à 5 jours par an, quel que soit le nombre d'enfants à charge, tant qu'ils ont moins de 16 ans. [F151]

❌ FAUX : la règle générale (3 jours par an) manque

### Avec le titre


Je ne sais pas.

❌ FAUX : la règle générale (3 jours par an) manque

### Avec le titre et la section entière


La durée maximale du congé pour enfant malade est fixée à 3 jours par an dans votre cas, car vous avez deux enfants à charge et cela correspond à la catégorie "Autre cas" mentionnée dans les extraits. [F151]

✅ JUSTE : la règle générale (3 jours par an) est dans la réponse

Chunks envoyés : 3 → 4 | ≈ 325 tokens de texte


- **Sans titre** : la preuve n'est **même pas dans le top 20** → « 5 jours ». **Faux, sourcé, sûr de lui.**
- **Avec le titre** : elle est **5e**, hors des 3 envoyés → « Je ne sais pas ».
- **Avec la section entière** : ✅ **juste**, 3 → 4 chunks, ≈ 325 tokens de texte.
- **Le titre fait remonter, la section donne le contexte** : les deux se complètent.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. Réparer avec BM25 </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **BM25** = recherche par **mots-clés**.
- **Fusion RRF** : chaque chunk reçoit 1 / (60 + rang) dans chaque classement, on additionne.

```
dense :  … 5e   ┐
                ├── fusion ──▶  3e   →  envoyé au modèle
BM25  :  1er    ┘
```

In [9]:
def tokenize(text):
    """Lowercase words, hyphens kept (L3142-4 stays one token)."""
    return re.findall(r"[\w-]+", text.lower())


bm25 = BM25Okapi([tokenize(chunk["text"]) for chunk in chunks])


def search_hybrid(question, k=3):
    """Fuses the dense and BM25 rankings with reciprocal rank fusion."""
    dense_ranking = search(question, k=50)
    bm25_ranking = list(np.argsort(-bm25.get_scores(tokenize(question)))[:50])
    fused = {}
    for ranking in (dense_ranking, bm25_ranking):
        for rank, position in enumerate(ranking, start=1):
            fused[position] = fused.get(position, 0) + 1 / (60 + rank)
    return sorted(fused, key=lambda position: -fused[position])[:k]


bm25_ranking = list(np.argsort(-bm25.get_scores(tokenize(QUESTION)))[:20])
print("Rang de la réponse, BM25 seul :", rank_of_answer(bm25_ranking))
print("Rang de la réponse, hybride   :", rank_of_answer(search_hybrid(QUESTION, k=20)))
print()
show(answer(QUESTION, search_hybrid(QUESTION)))

Rang de la réponse, BM25 seul : 1


Rang de la réponse, hybride   : 3



La durée maximale du congé pour enfant malade est fixée à 5 jours par an si l’enfant du salarié a moins d'1 an ou si le salarié a au moins 3 enfants à charge de moins de 16 ans. Dans d'autres cas, la durée maximale est de 3 jours par an. La convention collective peut accorder une durée de congé pour enfant malade plus longue. [F151]

✅ JUSTE : la règle générale (3 jours par an) est dans la réponse


- BM25 : **1er** ; hybride : **3e** → réponse **juste**.
- BM25 brille aussi sur les **codes exacts** (`L3142-4`).
- En production, **Qdrant fait la fusion** lui-même (Part 13).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 6. Un reranker heuristique, avec un seuil </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Un reranker **n'est pas forcément un modèle**.
- Score = **dense + 0,5 × part des mots de la question** présents dans le chunk.
- **Seuil de 0,45** : un chunk sous le seuil n'est pas envoyé. Aucun au-dessus → « Je ne sais pas », **sans appeler le modèle**.

In [10]:
STOPWORDS = set("le la les l un une des de du d à au aux en et ou pour par sur dans mon ma mes son sa ses "
                "que qui quoi quel quelle est il elle je j on se s ce cette peut peut-il combien comment y a ai-je".split())


def keywords(text):
    """Words of a text, lowercased, without the most frequent French words."""
    return {word for word in tokenize(text) if word not in STOPWORDS and len(word) > 1}


def search_heuristic(question, k=3, weight=0.5, threshold=0.45):
    """Reorders the dense top 20 with a keyword bonus, and drops chunks under the threshold."""
    query = np.array(embeddings.embed_query(question))
    scores = vectors @ (query / np.linalg.norm(query))
    question_words = keywords(question)
    results = []
    for position in np.argsort(-scores)[:20]:
        if scores[position] < threshold:
            continue
        shared = question_words & keywords(chunks[position]["text"])
        results.append((scores[position] + weight * len(shared) / len(question_words), position))
    results.sort(reverse=True)
    return [position for _, position in results[:k]]


print("Rang de la réponse, heuristique :", rank_of_answer(search_heuristic(QUESTION, k=20)))
print()
show(answer(QUESTION, search_heuristic(QUESTION)))

Rang de la réponse, heuristique : 3



La durée maximale du congé pour enfant malade est fixée à 5 jours par an si l’enfant a moins d'1 an ou si le salarié a au moins 3 enfants à charge de moins de 16 ans. Dans d'autres cas, la durée maximale est de 3 jours par an. [F151]

✅ JUSTE : la règle générale (3 jours par an) est dans la réponse


- Le **seuil**, sur une question hors sujet.

In [11]:
for question in ["Donne-moi une recette de crêpes.", "Combien de jours de congé pour mon mariage ?"]:
    query = np.array(embeddings.embed_query(question))
    best_score = max(vectors @ (query / np.linalg.norm(query)))
    positions = search_heuristic(question)
    print(question, "| meilleur score dense :", round(float(best_score), 2))
    if not positions:
        print("   aucun chunk au-dessus du seuil : Je ne sais pas.")
    else:
        print("  ", answer(question, positions))

Donne-moi une recette de crêpes. | meilleur score dense : 0.22
   aucun chunk au-dessus du seuil : Je ne sais pas.


Combien de jours de congé pour mon mariage ? | meilleur score dense : 0.72


   La durée de votre congé est de 4 jours. [F34154]


- Sur les **14 questions de test** : le bon chunk est-il dans les 3 envoyés ?

In [12]:
dense_hits = heuristic_hits = 0
for question, needle in TEST_SET:
    if (rank_of_answer(search(question), needle) or 99) <= 3:
        dense_hits += 1
    if (rank_of_answer(search_heuristic(question), needle) or 99) <= 3:
        heuristic_hits += 1
print("Bon chunk dans les 3 premiers : dense", dense_hits, "/ 14 | heuristique", heuristic_hits, "/ 14")

Bon chunk dans les 3 premiers : dense 12 / 14 | heuristique 14 / 14


- Enfant malade : **5e → 3e**, réponse juste.
- Top 3 : **dense 12/14 → heuristique 14/14**.
- Hors sujet (0,22) **écarté** par le seuil ; les vraies questions dépassent 0,7.
- ⚠️ Poids et seuil **réglés sur ces questions** : à revérifier sur un autre corpus.
- **Gratuit, instantané, explicable** : un bon premier palier.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 7. Réparer avec un reranker </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Un **reranker** lit **la question et le chunk ensemble** → plus précis, plus lent.
- On l'applique aux **20 meilleurs** candidats seulement.
- `BAAI/bge-reranker-v2-m3` : gratuit, multilingue, **local**.

In [13]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")


def search_reranked(question, k=3, candidates=20):
    """Reorders the dense top candidates with the reranker."""
    positions = search(question, k=candidates)
    pairs = [(question, chunks[position]["text"]) for position in positions]
    scores = reranker.predict(pairs)
    order = np.argsort(-scores)
    return [positions[i] for i in order[:k]]


start = time.perf_counter()
reranked = search_reranked(QUESTION, k=20)
print("Rang de la réponse, avec reranker :", rank_of_answer(reranked), f"({time.perf_counter() - start:.1f} s)")
print()
show(answer(QUESTION, reranked[:3]))

Rang de la réponse, avec reranker : 1 (0.9 s)



La durée maximale du congé pour enfant malade est fixée à 3 jours par an si l'enfant a plus d'un an, et à 5 jours par an si l'enfant a moins d'un an ou si le salarié a au moins 3 enfants à charge de moins de 16 ans. [F151]

✅ JUSTE : la règle générale (3 jours par an) est dans la réponse


![](assets/08_repairs.png)

- Reranker : **1er**, réponse juste.
- Le prix : **≈ 1 s** sur GPU, **≈ 15 s** sur CPU, contre quelques millisecondes en dense.
- En production : un **service** (Cohere, Jina) ou un GPU.
- ⚠️ Mesuré sur 16 autres questions : le reranker **améliore le classement, pas la qualité des réponses**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 8. Le contextual retrieval </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **Contextual retrieval** (Anthropic, 2024) : une phrase générée par un LLM **situe chaque chunk** dans sa fiche, avant la vectorisation.
- Mesuré **avant / après** sur les 14 questions de test.

```
« ### Autre cas … 3 jours par an »
                 ▼
« Ce passage porte sur la durée du congé pour enfant malade…  ### Autre cas … 3 jours par an »
```

In [14]:
CONTEXT_PROMPT = """<document>
{document}
</document>
Voici un passage de ce document :
<passage>
{chunk}
</passage>
Écrivez une phrase courte qui situe ce passage dans le document (sujet de la fiche et section), pour améliorer la recherche de ce passage. Répondez par la phrase seule."""

prompts = []
for chunk in chunks:
    prompts.append(CONTEXT_PROMPT.format(document=docs[chunk["doc_id"]], chunk=chunk["text"]))

start = time.perf_counter()
responses = llm.batch(prompts, config={"max_concurrency": 16})
contexts = [response.content.strip() for response in responses]

input_tokens = sum(response.usage_metadata["input_tokens"] for response in responses)
cached_tokens = sum(response.usage_metadata.get("input_token_details", {}).get("cache_read", 0) for response in responses)
print(len(contexts), "contextes générés en", round(time.perf_counter() - start), "s")
print("tokens d'entrée :", input_tokens, "dont en cache :", cached_tokens)

example = [i for i, chunk in enumerate(chunks) if NEEDLE in chunk["text"] and chunk["doc_id"] == "F151"][0]
print("\nContexte du chunk de la réponse :", contexts[example])

190 contextes générés en 13 s
tokens d'entrée : 683404 dont en cache : 547328

Contexte du chunk de la réponse : Ce passage se situe dans la section sur la durée du congé pour enfant malade et les droits du salarié concernant la prise de ce congé.


In [15]:
contextual_vectors = np.array(embeddings.embed_documents(
    [contexts[i] + "\n\n" + chunk["text"] for i, chunk in enumerate(chunks)]))
contextual_vectors = contextual_vectors / np.linalg.norm(contextual_vectors, axis=1, keepdims=True)



def rank_in(matrix, question, needle):
    """Rank of the first chunk containing needle, searching in matrix."""
    query = np.array(embeddings.embed_query(question))
    order = np.argsort(-(matrix @ (query / np.linalg.norm(query))))
    for rank, position in enumerate(order[:20], start=1):
        if needle.lower() in chunks[position]["text"].lower():
            return rank
    return None


better = worse = 0
for question, needle in TEST_SET:
    before = rank_in(vectors, question, needle)
    after = rank_in(contextual_vectors, question, needle)
    if after < before:
        better += 1
    if after > before:
        worse += 1
    print(f"avant {before:>2}   après {after:>2}   {question}")
print(f"\n{better} questions améliorées, {worse} dégradées, {len(TEST_SET) - better - worse} inchangées")

avant  1   après  3   Que prévoit l'article L3142-4 du Code du travail ?


avant  1   après  1   Combien de jours de congés payés acquiert-on par mois travaillé ?


avant  1   après  1   Combien de jours de congé pour mon mariage ?


avant  1   après  1   Combien de jours de congé si mon père décède ?


avant  1   après  1   Mon employeur peut-il m'imposer mes dates de congés ?


avant  3   après  1   Quelle est la période légale de prise des congés payés ?


... (4 outputs left out)

avant  1   après  2   Quelle ancienneté faut-il pour un congé sabbatique ?


avant  5   après  5   Combien de jours de congé pour enfant malade ?


avant  5   après  5   Le congé pour enfant malade est-il payé ?


avant  1   après  1   Combien de jours de congé pour le mariage de mon enfant ?

2 questions améliorées, 2 dégradées, 10 inchangées


- Indexation : **un appel au LLM par chunk**, mais **80 à 95 % des tokens d'entrée en cache** (même fiche en tête du prompt) → Part 12.
- Résultat : **autant de questions dégradées qu'améliorées**, la plupart inchangées (le détail varie d'une exécution à l'autre). L'enfant malade ne gagne rien.
- Le **titre** (section 3, gratuit) : 3 questions gagnées, 1 perdue. Le contexte **généré** (190 appels) : autant gagnées que perdues. **Commencez par le titre.**
- **Une technique à la mode ne paie pas forcément sur votre corpus : on mesure.** → Part 9, Lab 5.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > Récapitulatif </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| Technique vue dans la démo | Verdict | Où vous la mettez en pratique |
|---|---|---|
| **Regarder ce que le modèle a reçu** (la preuve est-elle dans le contexte ?) | indispensable | **Lab 4** (section 4) |
| **Le titre devant le chunk** | gratuit, le meilleur premier réglage | **Lab 4** (section 5) |
| **La section entière à la place du chunk** | règle les cas où le chunk ne dit rien seul | **Lab 4** (section 6) |
| **BM25 et fusion RRF** | utile sur les codes exacts, pas un gain systématique | **Projet, chapitre 5** |
| **Reranker heuristique + seuil** | gratuit, refuse le hors-sujet, mais réglé sur nos questions | **Projet, chapitre 5** |
| **Sous-questions** (non montrées ici) | à combiner avec le reste | **Projet, chapitre 5** |
| **Reranker** | meilleur classement, pas de meilleures réponses | **Projet, chapitre 5**, en option |
| **Contextual retrieval** | coûteux : un appel par chunk | **Démo seulement** ; le cache → **Part 12** |
| **Mesurer** (jeu de questions, notation, bruit) | ce qui permet de choisir entre tout ça | **Part 9 et Lab 5**, puis **chapitre 7** |

- **Retenez l'ordre** : regarder le contexte → **titre** → **section entière** → le reste, **mesuré**.
- **Prochain pas : le Lab 4.**